In [62]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [109]:
df = pd.read_csv(r'C:\Users\PL\Documents\ML data\diamonds.csv')

In [110]:
df.head()

,Unnamed: 0,carat,cut,color,clarity,depth,table,price,x,y,z
0,1,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,2,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,3,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,4,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,5,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


In [111]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53940 entries, 0 to 53939
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  53940 non-null  int64  
 1   carat       53940 non-null  float64
 2   cut         53940 non-null  object 
 3   color       53940 non-null  object 
 4   clarity     53940 non-null  object 
 5   depth       53940 non-null  float64
 6   table       53940 non-null  float64
 7   price       53940 non-null  int64  
 8   x           53940 non-null  float64
 9   y           53940 non-null  float64
 10  z           53940 non-null  float64
dtypes: float64(6), int64(2), object(3)
memory usage: 4.5+ MB


In [112]:
df.describe()

,Unnamed: 0,carat,depth,table,price,x,y,z
count,53940.000000,53940.000000,53940.000000,53940.000000,53940.000000,53940.000000,53940.000000,53940.000000
mean,26970.500000,0.797940,61.749405,57.457184,3932.799722,5.731157,5.734526,3.538734
std,15571.281097,0.474011,1.432621,2.234491,3989.439738,1.121761,1.142135,0.705699
min,1.000000,0.200000,43.000000,43.000000,326.000000,0.000000,0.000000,0.000000
25%,13485.750000,0.400000,61.000000,56.000000,950.000000,4.710000,4.720000,2.910000
50%,26970.500000,0.700000,61.800000,57.000000,2401.000000,5.700000,5.710000,3.530000
75%,40455.250000,1.040000,62.500000,59.000000,5324.250000,6.540000,6.540000,4.040000
max,53940.000000,5.010000,79.000000,95.000000,18823.000000,10.740000,58.900000,31.800000


In [113]:
df = df[df['y'] < 30]
df = df[df['x'] > 0]
df = df[df['y'] > 0]
df = df[df['z'] > 0]

In [114]:
X = df[[
       'carat',
       'cut',
       'color',
       'clarity',
       'depth',
       'table']]
y = df['price']

X = pd.get_dummies(
    X,columns=['cut','color','clarity'],
    drop_first=True)

In [115]:
np.log(y)
y = np.log(y)

In [116]:
X.isna().sum()

carat            0
depth            0
table            0
cut_Good         0
cut_Ideal        0
cut_Premium      0
cut_Very Good    0
color_E          0
color_F          0
color_G          0
color_H          0
color_I          0
color_J          0
clarity_IF       0
clarity_SI1      0
clarity_SI2      0
clarity_VS1      0
clarity_VS2      0
clarity_VVS1     0
clarity_VVS2     0
dtype: int64

In [117]:
X_train, X_test, y_train, y_test = train_test_split (
    X, y, test_size = 0.25, random_state = 42 )

In [121]:
model = LinearRegression()

model.fit(X_train, y_train)
preds = model.predict(X_test)

preds_actual = np.exp(preds)

In [127]:
print(len(y_test), len(preds))
print(np.exp(preds).max())
print(np.exp(preds).min())

13480 13480
7110027.098015412
415.4958239606912


In [128]:
preds_exp = np.exp(preds)
print(preds_exp[preds_exp > 50000])

[  56572.63005187   53901.68989771   53232.76868527   68781.00504599
  103483.79192299  121559.71434756   50714.08400226   59923.79138993
   66835.45175178   54007.30217381   50391.90821569   60169.45399601
   54510.07665304   62653.88494096  640214.56268641   62722.70017367
   58081.3986681    66243.70174675   50696.97561678   67785.46617412
 2304630.8169352    98636.80301608  188696.61104847   51231.09654727
   50037.11599898   52446.0682085    53693.49114942   74015.40907788
   96188.13681019   74121.63056416   72059.52050211   52901.71974188
   81574.61922174   62432.46987163   87326.70966968   54479.86901316
   71287.42917937   65768.81820885  167073.21679936 7110027.09801541
   92230.53121397   79375.22639783   75709.62752021  108012.50167864
  250501.42709633   93417.03935467   56563.56742798   52990.52143932
   79434.8982324    52978.6172229    58976.94007299   90964.02724825
   68639.5643164    63959.916166     95554.64242206   60987.73764272
   61319.97981402  160395.87103677

In [129]:
print(np.exp(y_test).max())

18786.999999999996


In [135]:
y_test_actual = np.exp(y_test)
mse = mean_squared_error(y_test_actual, preds_actual)
rmse = mse**0.5
r2 = r2_score(y_test_actual, preds_actual)
preds_capped = np.clip(np.exp(preds), 0, 20000)
print(mean_squared_error(np.exp(y_test), preds_capped)**0.5)
print(r2_score(np.exp(y_test), preds_capped))

1653.9500116536908
0.8272427578441506


In [55]:
df['clarity'].unique()

array(['SI2', 'SI1', 'VS1', 'VS2', 'VVS2', 'VVS1', 'I1', 'IF'],
      dtype=object)

In [134]:
coeffs = pd.Series(model.coef_,index=X.columns)
coeffs.sort_values(ascending = False) 

carat            2.197664
clarity_IF       1.043844
clarity_VVS1     0.956948
clarity_VVS2     0.942512
clarity_VS1      0.897893
clarity_VS2      0.832847
clarity_SI1      0.738373
clarity_SI2      0.551734
cut_Ideal        0.096474
cut_Very Good    0.059271
cut_Premium      0.054679
cut_Good         0.048817
table            0.005626
depth           -0.000504
color_F         -0.050737
color_E         -0.057926
color_G         -0.130353
color_H         -0.265753
color_I         -0.419101
color_J         -0.576223
dtype: float64

In [57]:
print(model.intercept_)

1.6717150747764924


In [58]:
print(X_test.iloc[1].sort_values(ascending=False))

depth             60.0
table             57.0
clarity_VVS2      True
cut_Very Good     True
color_F           True
carat             0.58
cut_Premium      False
color_E          False
cut_Ideal        False
color_G          False
cut_Good         False
color_I          False
color_J          False
clarity_IF       False
clarity_SI1      False
clarity_SI2      False
clarity_VS1      False
clarity_VS2      False
clarity_VVS1     False
color_H          False
Name: 50052, dtype: object


In [59]:
print(model.predict(X_test.iloc[[1]]))

[2.01384411]


In [60]:
df_results = pd.DataFrame({
    'actual': y_test,
    'predicted': preds,
    'error': y_test - preds})

In [61]:
df_results = df_results.reindex(df_results['error'].abs().sort_values(ascending=False).index)
df_results.head(10)

,actual,predicted,error
25998,2.264942,2.808083,-0.543141
26444,2.269994,2.802476,-0.532482
25999,2.264942,2.789591,-0.524649
27679,2.286083,2.749438,-0.463355
24328,2.245000,2.687996,-0.442996
21862,2.219509,2.572680,-0.353171
26932,2.276159,2.592768,-0.316609
27016,2.277218,2.559758,-0.282540
25778,2.261652,2.536160,-0.274508
25900,2.263616,2.537156,-0.273540
